# TestBED: Introduction and Examples
Author: Gavin Kerrigan (gavin.kerrigan@stats.ox.ac.uk)

Date: 2025 August 08

This notebook serves to showcase some of the basic functionality of the TestBED package.

In [1]:
import torch

---

## Tasks

The first object we will create is a `Task`. This is the object which implements the model which you are doing experimental design for. That is, it implements a prior $p(\theta)$ and likelihood $p(y \mid \theta, \xi)$. For many problems, you will be able to evaluate this likelihood, but for implicit problems, you can only sample $y \sim p(y \mid \theta, \xi)$. TestBED supports both kinds of models.

Some useful examples of things you can do with any `Task` include:
- `sample_prior(n)` will sample $\theta_i \sim p(\theta)$.
- `sample_joint(n, design)` will sample $(y_i, \theta_i) \sim p(y, \theta \mid \xi)$.
- For explicit models, `outcome_logp(y, theta, design)` will evaluate $\log p(y_i \mid \theta_i, \xi_i)$.

The classic example is the `LocationFinding` task. Creating the task in this way will create an instance of this task with some default hyperparameters defining the likelihood as well as a standard Gaussian prior.

In [2]:
from testbed.tasks import LocationFinding

n_sources = 4
dim_theta = 2
task = LocationFinding(n_sources, dim_theta)

design = torch.randn((1, 2))
y, theta = task.sample_joint(n_samples=100, design=design)
print(f'y shape: {y.shape} theta shape: {theta.shape}')

y shape: torch.Size([100, 1]) theta shape: torch.Size([100, 4, 2])


You can of course also further specify the hyperparameters and prior in this model. `LocationFinding` already has a convenient way of specifying an independent Gaussian prior for each source, but it is easy enough to implement your own if you so desire.

In [3]:
from testbed.tasks.location_finding import LocationFindingPrior

# Setting up a new prior
prior_mean = torch.randn((n_sources, dim_theta))
prior_cov = 2. * torch.eye(dim_theta)
prior = LocationFindingPrior(n_sources, dim_theta, 
                             theta_loc=prior_mean, theta_covmat=prior_cov)

# Creating a LocationFinding task with the new prior and likelihood settings
task = LocationFinding(n_sources, dim_theta, prior=prior,
                       base_signal=0.3, max_signal=1e-2, noise_scale=0.25)

design = torch.randn((1, 2))
y, theta = task.sample_joint(n_samples=100, design=design)
print(f'y shape: {y.shape} theta shape: {theta.shape}')


y shape: torch.Size([100, 1]) theta shape: torch.Size([100, 4, 2])


---

##  Designers and Objectives

A `Designer` is an object which produces new designs for a given `Task`. A `Designer` can be (almost) anything you want, as long as it outputs valid designs. Typically, a design will be selected by optimizing an `Objective`.

The most common types of `Designers` are:
 - `SequentialDesigner`s which greedily optimize the `Objective` at each step of the design process.
  - `PolicyDesigner`s, which learn neural networks to amoritze the design process.


While there are key differences between different kinds of designers (e.g., how they are fit, and the mechanisms behind how they propose designs), the abstract `Designer` class serves to unify some of the common functionality. Most notably, all `Designer` objects implement:
 - `propose_design(y_history, design_history)`, which takes in the history of the design process and outputs a single new design.
 - `rollout(theta)`, which rolls out the designer to produce a sequence of designs given a true value of `theta`.


Here, we demonstrate how to set up a `PCE` objective with a `SequentialDesigner`. Let's first set up our objective. 

In [4]:
from testbed.design_objectives import PCE

In [5]:
n_sources = 4
dim_theta = 2
task = LocationFinding(n_sources, dim_theta)

objective = PCE(task, n_outer=100, n_inner=100)

design = torch.randn((1, 2))
eig = objective.estimate_gain(design)
print(f'Estimated EIG value: {eig}')

Estimated EIG value: 0.7330107688903809


In [6]:
from testbed.designers import SequentialDesigner

In [ ]:
designer = SequentialDesigner(task, objective, T=5)

In [11]:
init_design = torch.randn((1, 2), requires_grad=True)
lr = 1e-3
n_step_opt = 250

new_design = designer.propose_design(init_design, lr, n_step_opt)
new_design

tensor([[-0.6651,  0.2021]], requires_grad=True)

## ToDos:
- Implement sequential propose-update steps in designer 
- Actually implement inference
- Have a nicer way of handling the hyperparams of design proposal (initialization, optimizer args...)
- Docs throughout
- Logging throughout
- Extensive testing for correctness 